In [3]:
import numpy as np
import matplotlib.pyplot as plt

In [4]:
# Two similar vectors (small area)
v1 = np.array([1, 0])
v2 = np.array([0.9, 0.1])
similar_matrix = np.array([v1, v2])
print(f"Similar vectors determinant: {np.linalg.det(similar_matrix):.3f}")

Similar vectors determinant: 0.100


In [5]:
# Two orthogonal vectors (maximum area)
v1 = np.array([1, 0])
v2 = np.array([0, 1])
diverse_matrix = np.array([v1, v2])
print(f"Orthogonal vectors determinant: {np.linalg.det(diverse_matrix):.3f}")

Orthogonal vectors determinant: 1.000


In [6]:
relevance = np.array([0.9, 0.85, 0.8, 0.7, 0.6, 0.5])
genre = ['Action', 'Action', 'Action', 'Comedy', 'Documentary', 'Mixed comedy/documentary']
embeddings = np.array([
    [1.0, 0.0, 0.0],   # video 0: pure action
    [0.9, 0.1, 0.0],   # video 1: action (very similar to 0)
    [0.8, 0.1, 0.1],   # video 2: action (also very similar to 0)
    [0.0, 1.0, 0.0],   # video 3: comedy
    [0.0, 0.0, 1.0],   # video 4: documentary
    [0.1, 0.5, 0.5],   # video 5: mixed
])
# Build kernel matrix L = VVᵀ (Gram matrix from embeddings)
L = embeddings @ embeddings.T
print("Kernel matrix L:")
print(np.round(L, 2))
print(f"\nSimilarity of video 0 ({genre[0]}), with other videos")
for i in range(1,len(genre)):
    print(f"Video {i}: {L[0][i]} ({genre[i]})")

Kernel matrix L:
[[1.   0.9  0.8  0.   0.   0.1 ]
 [0.9  0.82 0.73 0.1  0.   0.14]
 [0.8  0.73 0.66 0.1  0.1  0.18]
 [0.   0.1  0.1  1.   0.   0.5 ]
 [0.   0.   0.1  0.   1.   0.5 ]
 [0.1  0.14 0.18 0.5  0.5  0.51]]

Similarity of video 0 (Action), with other videos
Video 1: 0.9 (Action)
Video 2: 0.8 (Action)
Video 3: 0.0 (Comedy)
Video 4: 0.0 (Documentary)
Video 5: 0.1 (Mixed comedy/documentary)


In [7]:
def subset_det(L, indices):
    """Compute det(L_Y) for a subset of indices"""
    L_Y = L[np.ix_(indices, indices)]
    return np.linalg.det(L_Y)

# Set 1: Three action videos (redundant)
redundant = [0, 1, 2]  # Action, Action, Action
det_redundant = subset_det(L, redundant)
print(f"Redundant set: {[genre[i] for i in redundant]}")
print(f"  det(L_Y) = {det_redundant:.4f}")
# Set 2: Action + Comedy + Documentary (diverse)
diverse = [0, 3, 4]
det_diverse = subset_det(L, diverse)
print(f"\nDiverse set: {[genre[i] for i in diverse]}")
print(f"  det(L_Y) = {det_diverse:.4f}")
print(f"\nDiverse set is {det_diverse/det_redundant:.0f}x more likely under DPP")

Redundant set: ['Action', 'Action', 'Action']
  det(L_Y) = 0.0001

Diverse set: ['Action', 'Comedy', 'Documentary']
  det(L_Y) = 1.0000

Diverse set is 10000x more likely under DPP


In [8]:
# Similarity matrix from embeddings
S = embeddings @ embeddings.T

# Quality-weighted kernel: L[i,j] = q_i * q_j * S[i,j]
q = relevance.reshape(-1, 1)  # Column vector
L_quality = (q @ q.T) * S
print("Quality-weighted kernel L:")
print(np.round(L_quality, 3))

Quality-weighted kernel L:
[[0.81  0.688 0.576 0.    0.    0.045]
 [0.688 0.592 0.496 0.06  0.    0.06 ]
 [0.576 0.496 0.422 0.056 0.048 0.072]
 [0.    0.06  0.056 0.49  0.    0.175]
 [0.    0.    0.048 0.    0.36  0.15 ]
 [0.045 0.06  0.072 0.175 0.15  0.128]]


In [9]:
redundant = [0, 1, 2]  # Top 3 by relevance, all Action
diverse = [0, 3, 4]    # Action + Comedy + Documentary

det_redundant = subset_det(L_quality, redundant)
det_diverse = subset_det(L_quality, diverse)
print(f"Redundant set (top-3 relevance): {[f'{genre[i]} ({relevance[i]})' for i in redundant]}")
print(f" det = {det_redundant:.6f}")
print(f"\nDiverse set (mixed relevance): {[f'{genre[i]} ({relevance[i]})' for i in diverse]}")
print(f" det = {det_diverse:.6f}")
print(f"\nDiverse set is {det_diverse/det_redundant:.0f}x more likely")

Redundant set (top-3 relevance): ['Action (0.9)', 'Action (0.85)', 'Action (0.8)']
 det = 0.000037

Diverse set (mixed relevance): ['Action (0.9)', 'Comedy (0.7)', 'Documentary (0.6)']
 det = 0.142884

Diverse set is 3815x more likely


In [146]:
np.ix_([0, 1], [0, 1])

(array([[0],
        [1]]),
 array([[0, 1]]))

In [150]:
L

array([[0.7225    , 0.50492542, 0.15396552, 0.18855827],
       [0.50492542, 0.64      , 0.17348724, 0.20009304],
       [0.15396552, 0.17348724, 0.36      , 0.17435621],
       [0.18855827, 0.20009304, 0.17435621, 0.25      ]])

In [149]:
L[np.ix_([0, 1], [0, 1])]

array([[0.7225    , 0.50492542],
       [0.50492542, 0.64      ]])

In [160]:
np.linalg.det(L_quality[np.ix_([0, 1], [0, 1])])

np.float64(0.005852250000000031)

In [167]:
temp = np.random.rand(5, 2, 2)
temp

array([[[0.76396443, 0.79965728],
        [0.0746352 , 0.82679416]],

       [[0.83780378, 0.02413593],
        [0.1468246 , 0.24364947]],

       [[0.3020599 , 0.46568976],
        [0.06637677, 0.79866974]],

       [[0.41238848, 0.44212124],
        [0.89458274, 0.60989198]],

       [[0.03294311, 0.76292009],
        [0.20661959, 0.08758741]]])

In [168]:
np.linalg.det(temp)

array([ 0.57195875,  0.2005867 ,  0.21033512, -0.1440016 , -0.15474883])

In [169]:
np.linalg.det(temp[0])

np.float64(0.5719587508539751)

In [159]:
np.linalg.det(L_quality[np.ix_([0, 3], [0, 3])])

np.float64(0.3969)

In [152]:
genre

['Action',
 'Action',
 'Action',
 'Comedy',
 'Documentary',
 'Mixed comedy/documentary']

In [157]:
stuff = set()
stuff.add(3)
stuff.add(1)
np.array(list(stuff))

array([1, 3])

In [175]:
L_quality

array([[0.81   , 0.6885 , 0.576  , 0.     , 0.     , 0.045  ],
       [0.6885 , 0.59245, 0.4964 , 0.0595 , 0.     , 0.0595 ],
       [0.576  , 0.4964 , 0.4224 , 0.056  , 0.048  , 0.072  ],
       [0.     , 0.0595 , 0.056  , 0.49   , 0.     , 0.175  ],
       [0.     , 0.     , 0.048  , 0.     , 0.36   , 0.15   ],
       [0.045  , 0.0595 , 0.072  , 0.175  , 0.15   , 0.1275 ]])

In [177]:
items = set(range(6))
items

{0, 1, 2, 3, 4, 5}

In [178]:
np.array(list(items))

array([0, 1, 2, 3, 4, 5])

In [173]:
np.ix_([0, 1], [0, 1])

(array([[0],
        [1]]),
 array([[0, 1]]))

In [184]:
np.meshgrid([0, 1], [0, 1, 2, 3, 4])

(array([[0, 1],
        [0, 1],
        [0, 1],
        [0, 1],
        [0, 1]]),
 array([[0, 0],
        [1, 1],
        [2, 2],
        [3, 3],
        [4, 4]]))

In [186]:
L_quality[np.meshgrid([0, 3], [0, 3])]

array([[0.81, 0.  ],
       [0.  , 0.49]])

In [187]:
np.meshgrid([0], [0])

(array([[0]]), array([[0]]))

In [ ]:
from itertools import combinations

list(combinations(items, 1))

[(0,), (1,), (2,), (3,), (4,), (5,)]

In [221]:
np.meshgrid([0], [0])

(array([[0]]), array([[0]]))

In [226]:
np.meshgrid([0, 8], [0, 8])

(array([[0, 8],
        [0, 8]]),
 array([[0, 0],
        [8, 8]]))

In [224]:
np.array(list(combinations(items, 1)))

array([[0],
       [1],
       [2],
       [3],
       [4],
       [5]])

In [227]:
np.array(list(combinations(items, 1))).T

array([[0, 1, 2, 3, 4, 5]])

In [237]:
L_quality

array([[0.81   , 0.6885 , 0.576  , 0.     , 0.     , 0.045  ],
       [0.6885 , 0.59245, 0.4964 , 0.0595 , 0.     , 0.0595 ],
       [0.576  , 0.4964 , 0.4224 , 0.056  , 0.048  , 0.072  ],
       [0.     , 0.0595 , 0.056  , 0.49   , 0.     , 0.175  ],
       [0.     , 0.     , 0.048  , 0.     , 0.36   , 0.15   ],
       [0.045  , 0.0595 , 0.072  , 0.175  , 0.15   , 0.1275 ]])

In [236]:
L_quality[np.array(list(combinations(items, 1))), np.array(list(combinations(items, 1)))]

array([[0.81   ],
       [0.59245],
       [0.4224 ],
       [0.49   ],
       [0.36   ],
       [0.1275 ]])

In [240]:
np.array(list(combinations(items, 2)))

array([[0, 1],
       [0, 2],
       [0, 3],
       [0, 4],
       [0, 5],
       [1, 2],
       [1, 3],
       [1, 4],
       [1, 5],
       [2, 3],
       [2, 4],
       [2, 5],
       [3, 4],
       [3, 5],
       [4, 5]])

In [233]:
np.array(list(combinations(items, 2))).T

array([[0, 0, 0, 0, 0, 1, 1, 1, 1, 2, 2, 2, 3, 3, 4],
       [1, 2, 3, 4, 5, 2, 3, 4, 5, 3, 4, 5, 4, 5, 5]])

In [238]:
np.meshgrid([0, 1], [0, 1])

(array([[0, 1],
        [0, 1]]),
 array([[0, 0],
        [1, 1]]))

In [266]:
np.array([[0, 1], [0, 2], [1, 2]])[:, :, None]

array([[[0],
        [1]],

       [[0],
        [2]],

       [[1],
        [2]]])

In [267]:
np.array([[0, 1], [0, 2], [1, 2]])[:, None, :]

array([[[0, 1]],

       [[0, 2]],

       [[1, 2]]])

In [271]:
np.expand_dims(np.array([[0, 1], [0, 2], [1, 2]]), -1)

array([[[0],
        [1]],

       [[0],
        [2]],

       [[1],
        [2]]])

In [270]:
np.expand_dims(np.array([[0, 1], [0, 2], [1, 2]]), 1)

array([[[0, 1]],

       [[0, 2]],

       [[1, 2]]])

In [268]:
L_quality[np.array([[0, 1], [0, 2], [1, 2]])[:, :, None], np.array([[0, 1], [0, 2], [1, 2]])[:, None, :]]

array([[[0.81   , 0.6885 ],
        [0.6885 , 0.59245]],

       [[0.81   , 0.576  ],
        [0.576  , 0.4224 ]],

       [[0.59245, 0.4964 ],
        [0.4964 , 0.4224 ]]])

In [290]:
combs = np.array(list(combinations(items, 1)))

L_quality[np.expand_dims(combs, -1), np.expand_dims(combs, 1)]

array([[[0.81   ]],

       [[0.59245]],

       [[0.4224 ]],

       [[0.49   ]],

       [[0.36   ]],

       [[0.1275 ]]])

In [ ]:
np.linalg.det(L_quality[np.expand_dims(combs, -1), np.expand_dims(combs, 1)])

array([0.81   , 0.59245, 0.4224 , 0.49   , 0.36   , 0.1275 ])

In [ ]:
np.argmax(np.linalg.det(L_quality[np.expand_dims(combs, -1), np.expand_dims(combs, 1)]))

np.int64(0)

In [278]:
from itertools import product

all_items = list(items)
list(product([all_items[0]], all_items[1:]))

[(0, 1), (0, 2), (0, 3), (0, 4), (0, 5)]

In [280]:
list(product([all_items[0], all_items[1]], all_items[2:]))

[(0, 2), (0, 3), (0, 4), (0, 5), (1, 2), (1, 3), (1, 4), (1, 5)]

In [279]:
list(product([], all_items[1:]))

[]

In [274]:
list(combinations(items, 2))

[(0, 1),
 (0, 2),
 (0, 3),
 (0, 4),
 (0, 5),
 (1, 2),
 (1, 3),
 (1, 4),
 (1, 5),
 (2, 3),
 (2, 4),
 (2, 5),
 (3, 4),
 (3, 5),
 (4, 5)]

In [294]:
combs = np.array(list(combinations(items, 2)))

L_quality[np.expand_dims(combs, -1), np.expand_dims(combs, 1)]

array([[[0.81   , 0.6885 ],
        [0.6885 , 0.59245]],

       [[0.81   , 0.576  ],
        [0.576  , 0.4224 ]],

       [[0.81   , 0.     ],
        [0.     , 0.49   ]],

       [[0.81   , 0.     ],
        [0.     , 0.36   ]],

       [[0.81   , 0.045  ],
        [0.045  , 0.1275 ]],

       [[0.59245, 0.4964 ],
        [0.4964 , 0.4224 ]],

       [[0.59245, 0.0595 ],
        [0.0595 , 0.49   ]],

       [[0.59245, 0.     ],
        [0.     , 0.36   ]],

       [[0.59245, 0.0595 ],
        [0.0595 , 0.1275 ]],

       [[0.4224 , 0.056  ],
        [0.056  , 0.49   ]],

       [[0.4224 , 0.048  ],
        [0.048  , 0.36   ]],

       [[0.4224 , 0.072  ],
        [0.072  , 0.1275 ]],

       [[0.49   , 0.     ],
        [0.     , 0.36   ]],

       [[0.49   , 0.175  ],
        [0.175  , 0.1275 ]],

       [[0.36   , 0.15   ],
        [0.15   , 0.1275 ]]])

In [295]:
np.linalg.det(L_quality[np.expand_dims(combs, -1), np.expand_dims(combs, 1)])

array([0.00585225, 0.010368  , 0.3969    , 0.2916    , 0.10125   ,
       0.00383792, 0.28676025, 0.213282  , 0.07199713, 0.20384   ,
       0.14976   , 0.048672  , 0.1764    , 0.03185   , 0.0234    ])

In [ ]:
np.argmax(np.linalg.det(L_quality[np.expand_dims(combs, -1), np.expand_dims(combs, 1)]))

np.int64(2)

In [ ]:
np.array(list(combinations(items, 1)))

In [243]:
L_quality

array([[0.81   , 0.6885 , 0.576  , 0.     , 0.     , 0.045  ],
       [0.6885 , 0.59245, 0.4964 , 0.0595 , 0.     , 0.0595 ],
       [0.576  , 0.4964 , 0.4224 , 0.056  , 0.048  , 0.072  ],
       [0.     , 0.0595 , 0.056  , 0.49   , 0.     , 0.175  ],
       [0.     , 0.     , 0.048  , 0.     , 0.36   , 0.15   ],
       [0.045  , 0.0595 , 0.072  , 0.175  , 0.15   , 0.1275 ]])

In [251]:
L_quality[np.array([[0, 1], [0, 2], [1, 2]])]

array([[[0.81   , 0.6885 , 0.576  , 0.     , 0.     , 0.045  ],
        [0.6885 , 0.59245, 0.4964 , 0.0595 , 0.     , 0.0595 ]],

       [[0.81   , 0.6885 , 0.576  , 0.     , 0.     , 0.045  ],
        [0.576  , 0.4964 , 0.4224 , 0.056  , 0.048  , 0.072  ]],

       [[0.6885 , 0.59245, 0.4964 , 0.0595 , 0.     , 0.0595 ],
        [0.576  , 0.4964 , 0.4224 , 0.056  , 0.048  , 0.072  ]]])

In [265]:
L_quality[np.array([[0, 1], [0, 2], [1, 2]])]

array([[[0.81   , 0.6885 , 0.576  , 0.     , 0.     , 0.045  ],
        [0.6885 , 0.59245, 0.4964 , 0.0595 , 0.     , 0.0595 ]],

       [[0.81   , 0.6885 , 0.576  , 0.     , 0.     , 0.045  ],
        [0.576  , 0.4964 , 0.4224 , 0.056  , 0.048  , 0.072  ]],

       [[0.6885 , 0.59245, 0.4964 , 0.0595 , 0.     , 0.0595 ],
        [0.576  , 0.4964 , 0.4224 , 0.056  , 0.048  , 0.072  ]]])

In [244]:
L_quality[np.array([[0, 1], [0, 2], [1, 2]]), np.array([[0, 1], [0, 2], [1, 2]])]

array([[0.81   , 0.59245],
       [0.81   , 0.4224 ],
       [0.59245, 0.4224 ]])

In [252]:
np.meshgrid([0, 1], [0, 1])

(array([[0, 1],
        [0, 1]]),
 array([[0, 0],
        [1, 1]]))

In [254]:
L_quality[np.meshgrid([0, 1], [0, 1])]

array([[0.81   , 0.6885 ],
       [0.6885 , 0.59245]])

In [255]:
L_quality[np.meshgrid([0, 2], [0, 2])]

array([[0.81  , 0.576 ],
       [0.576 , 0.4224]])

In [316]:
np.array(list(combinations(items, 1)))

array([[0],
       [1],
       [2],
       [3],
       [4],
       [5]])

In [315]:
L_quality[np.array(list(combinations(items, 1)))]

array([[[0.81   , 0.6885 , 0.576  , 0.     , 0.     , 0.045  ]],

       [[0.6885 , 0.59245, 0.4964 , 0.0595 , 0.     , 0.0595 ]],

       [[0.576  , 0.4964 , 0.4224 , 0.056  , 0.048  , 0.072  ]],

       [[0.     , 0.0595 , 0.056  , 0.49   , 0.     , 0.175  ]],

       [[0.     , 0.     , 0.048  , 0.     , 0.36   , 0.15   ]],

       [[0.045  , 0.0595 , 0.072  , 0.175  , 0.15   , 0.1275 ]]])

In [235]:
L_quality[np.array(list(combinations(items, 2))), np.array(list(combinations(items, 2)))]

array([[0.81   , 0.59245],
       [0.81   , 0.4224 ],
       [0.81   , 0.49   ],
       [0.81   , 0.36   ],
       [0.81   , 0.1275 ],
       [0.59245, 0.4224 ],
       [0.59245, 0.49   ],
       [0.59245, 0.36   ],
       [0.59245, 0.1275 ],
       [0.4224 , 0.49   ],
       [0.4224 , 0.36   ],
       [0.4224 , 0.1275 ],
       [0.49   , 0.36   ],
       [0.49   , 0.1275 ],
       [0.36   , 0.1275 ]])

(array([[0, 1, 2, 3, 4, 5],
        [0, 1, 2, 3, 4, 5],
        [0, 1, 2, 3, 4, 5],
        [0, 1, 2, 3, 4, 5],
        [0, 1, 2, 3, 4, 5],
        [0, 1, 2, 3, 4, 5]]),
 array([[0, 0, 0, 0, 0, 0],
        [1, 1, 1, 1, 1, 1],
        [2, 2, 2, 2, 2, 2],
        [3, 3, 3, 3, 3, 3],
        [4, 4, 4, 4, 4, 4],
        [5, 5, 5, 5, 5, 5]]))

In [203]:
[list(item) for item in combinations(items, 1)]

[[0], [1], [2], [3], [4], [5]]

In [204]:
np.meshgrid([list(item) for item in combinations(items, 1)], [list(item) for item in combinations(items, 1)])

(array([[0, 1, 2, 3, 4, 5],
        [0, 1, 2, 3, 4, 5],
        [0, 1, 2, 3, 4, 5],
        [0, 1, 2, 3, 4, 5],
        [0, 1, 2, 3, 4, 5],
        [0, 1, 2, 3, 4, 5]]),
 array([[0, 0, 0, 0, 0, 0],
        [1, 1, 1, 1, 1, 1],
        [2, 2, 2, 2, 2, 2],
        [3, 3, 3, 3, 3, 3],
        [4, 4, 4, 4, 4, 4],
        [5, 5, 5, 5, 5, 5]]))

In [207]:
list(combinations(items, 2))

[(0, 1),
 (0, 2),
 (0, 3),
 (0, 4),
 (0, 5),
 (1, 2),
 (1, 3),
 (1, 4),
 (1, 5),
 (2, 3),
 (2, 4),
 (2, 5),
 (3, 4),
 (3, 5),
 (4, 5)]

In [205]:
np.meshgrid([list(item) for item in combinations(items, 2)], [list(item) for item in combinations(items, 2)])

(array([[0, 1, 0, 2, 0, 3, 0, 4, 0, 5, 1, 2, 1, 3, 1, 4, 1, 5, 2, 3, 2, 4,
         2, 5, 3, 4, 3, 5, 4, 5],
        [0, 1, 0, 2, 0, 3, 0, 4, 0, 5, 1, 2, 1, 3, 1, 4, 1, 5, 2, 3, 2, 4,
         2, 5, 3, 4, 3, 5, 4, 5],
        [0, 1, 0, 2, 0, 3, 0, 4, 0, 5, 1, 2, 1, 3, 1, 4, 1, 5, 2, 3, 2, 4,
         2, 5, 3, 4, 3, 5, 4, 5],
        [0, 1, 0, 2, 0, 3, 0, 4, 0, 5, 1, 2, 1, 3, 1, 4, 1, 5, 2, 3, 2, 4,
         2, 5, 3, 4, 3, 5, 4, 5],
        [0, 1, 0, 2, 0, 3, 0, 4, 0, 5, 1, 2, 1, 3, 1, 4, 1, 5, 2, 3, 2, 4,
         2, 5, 3, 4, 3, 5, 4, 5],
        [0, 1, 0, 2, 0, 3, 0, 4, 0, 5, 1, 2, 1, 3, 1, 4, 1, 5, 2, 3, 2, 4,
         2, 5, 3, 4, 3, 5, 4, 5],
        [0, 1, 0, 2, 0, 3, 0, 4, 0, 5, 1, 2, 1, 3, 1, 4, 1, 5, 2, 3, 2, 4,
         2, 5, 3, 4, 3, 5, 4, 5],
        [0, 1, 0, 2, 0, 3, 0, 4, 0, 5, 1, 2, 1, 3, 1, 4, 1, 5, 2, 3, 2, 4,
         2, 5, 3, 4, 3, 5, 4, 5],
        [0, 1, 0, 2, 0, 3, 0, 4, 0, 5, 1, 2, 1, 3, 1, 4, 1, 5, 2, 3, 2, 4,
         2, 5, 3, 4, 3, 5, 4, 5],
        [0, 1, 0, 2

In [193]:
np.meshgrid(list(combinations(items, 1)), list(combinations(items, 1)))

(array([[0, 1, 2, 3, 4, 5],
        [0, 1, 2, 3, 4, 5],
        [0, 1, 2, 3, 4, 5],
        [0, 1, 2, 3, 4, 5],
        [0, 1, 2, 3, 4, 5],
        [0, 1, 2, 3, 4, 5]]),
 array([[0, 0, 0, 0, 0, 0],
        [1, 1, 1, 1, 1, 1],
        [2, 2, 2, 2, 2, 2],
        [3, 3, 3, 3, 3, 3],
        [4, 4, 4, 4, 4, 4],
        [5, 5, 5, 5, 5, 5]]))

In [198]:
L_quality[np.meshgrid(list(combinations(items, 2)), list(combinations(items, 2)))].shape

(30, 30)

In [188]:
np.meshgrid(np.array(list(items)), np.array(list(items)))

(array([[0, 1, 2, 3, 4, 5],
        [0, 1, 2, 3, 4, 5],
        [0, 1, 2, 3, 4, 5],
        [0, 1, 2, 3, 4, 5],
        [0, 1, 2, 3, 4, 5],
        [0, 1, 2, 3, 4, 5]]),
 array([[0, 0, 0, 0, 0, 0],
        [1, 1, 1, 1, 1, 1],
        [2, 2, 2, 2, 2, 2],
        [3, 3, 3, 3, 3, 3],
        [4, 4, 4, 4, 4, 4],
        [5, 5, 5, 5, 5, 5]]))

In [189]:
L_quality[np.meshgrid(np.array(list(items)), np.array(list(items)))]

array([[0.81   , 0.6885 , 0.576  , 0.     , 0.     , 0.045  ],
       [0.6885 , 0.59245, 0.4964 , 0.0595 , 0.     , 0.0595 ],
       [0.576  , 0.4964 , 0.4224 , 0.056  , 0.048  , 0.072  ],
       [0.     , 0.0595 , 0.056  , 0.49   , 0.     , 0.175  ],
       [0.     , 0.     , 0.048  , 0.     , 0.36   , 0.15   ],
       [0.045  , 0.0595 , 0.072  , 0.175  , 0.15   , 0.1275 ]])

In [190]:
L_quality

array([[0.81   , 0.6885 , 0.576  , 0.     , 0.     , 0.045  ],
       [0.6885 , 0.59245, 0.4964 , 0.0595 , 0.     , 0.0595 ],
       [0.576  , 0.4964 , 0.4224 , 0.056  , 0.048  , 0.072  ],
       [0.     , 0.0595 , 0.056  , 0.49   , 0.     , 0.175  ],
       [0.     , 0.     , 0.048  , 0.     , 0.36   , 0.15   ],
       [0.045  , 0.0595 , 0.072  , 0.175  , 0.15   , 0.1275 ]])

In [209]:
np.meshgrid([0, 3, 1], [0, 3, 1])

(array([[0, 3, 1],
        [0, 3, 1],
        [0, 3, 1]]),
 array([[0, 0, 0],
        [3, 3, 3],
        [1, 1, 1]]))

In [208]:
L_quality[np.meshgrid([0, 3, 1], [0, 3, 1])]

array([[0.81   , 0.     , 0.6885 ],
       [0.     , 0.49   , 0.0595 ],
       [0.6885 , 0.0595 , 0.59245]])

In [185]:
L_quality[np.ix_([0, 3], [0, 3])]

array([[0.81, 0.  ],
       [0.  , 0.49]])

In [314]:
def greedy_map_dpp_fr(L, k):
    N = L.shape[0]
    selected = []
    remaining = set(range(N))

    for _ in range(k):
        candidate_set = np.array([selected + [item] for item in remaining])

        det_arr = np.linalg.det(L[np.expand_dims(candidate_set, -1), np.expand_dims(candidate_set, 1)])
        best_item = candidate_set[np.argmax(det_arr).item()][-1]

        print(best_item)
        print(det_arr[np.argmax(det_arr).item()])
        print()
        selected.append(best_item)
        remaining.remove(best_item)

    return selected

# Run it
result = greedy_map_dpp_fr(L_quality, k=4)
print(f"Greedy MAP result: {[f'{genre[i]} ({relevance[i]})' for i in result]}")

0
0.81

3
0.3969

4
0.14288399999999996

2
7.931655332527042e-18

Greedy MAP result: ['Action (0.9)', 'Comedy (0.7)', 'Documentary (0.6)', 'Action (0.8)']


In [309]:
def greedy_map_dpp(L, k):
    N = L.shape[0]
    selected = []
    remaining = set(range(N))
    
    for _ in range(k):
        best_item, best_det = None, -1
        
        for item in remaining:
            candidate_set = selected + [item]
            #print(candidate_set)
            #print(np.ix_(candidate_set, candidate_set))
            #print(L[np.ix_(candidate_set, candidate_set)])
            
            det_val = np.linalg.det(L[np.ix_(candidate_set, candidate_set)])
            #print(det_val)
            #print()
            
            if det_val > best_det:
                
                best_det = det_val
                best_item = item
        print(best_item)
        print(best_det)
        print()
        
        selected.append(best_item)
        remaining.remove(best_item)
    
    return selected

# Run it
result = greedy_map_dpp(L_quality, k=4)
print(f"Greedy MAP result: {[f'{genre[i]} ({relevance[i]})' for i in result]}")

0
0.81

3
0.3969

4
0.14288399999999996

2
7.931655332527042e-18

Greedy MAP result: ['Action (0.9)', 'Comedy (0.7)', 'Documentary (0.6)', 'Action (0.8)']


In [ ]:
[], [0, 1, 2]

In [210]:
np.meshgrid([0, 1, 2], [0, 1, 2])

(array([[0, 1, 2],
        [0, 1, 2],
        [0, 1, 2]]),
 array([[0, 0, 0],
        [1, 1, 1],
        [2, 2, 2]]))

In [212]:
la = []
ra = [0, 1, 2]
r, c = np.triu_indices(len(ra), 1)
r, c

(array([0, 0, 1]), array([1, 2, 2]))

In [214]:
L_quality

array([[0.81   , 0.6885 , 0.576  , 0.     , 0.     , 0.045  ],
       [0.6885 , 0.59245, 0.4964 , 0.0595 , 0.     , 0.0595 ],
       [0.576  , 0.4964 , 0.4224 , 0.056  , 0.048  , 0.072  ],
       [0.     , 0.0595 , 0.056  , 0.49   , 0.     , 0.175  ],
       [0.     , 0.     , 0.048  , 0.     , 0.36   , 0.15   ],
       [0.045  , 0.0595 , 0.072  , 0.175  , 0.15   , 0.1275 ]])

In [220]:
np.vstack((r, c)).T

array([[0, 1],
       [0, 2],
       [1, 2]])

In [215]:
L_quality[np.meshgrid([0, 1, 2], [0, 1, 2])]

array([[0.81   , 0.6885 , 0.576  ],
       [0.6885 , 0.59245, 0.4964 ],
       [0.576  , 0.4964 , 0.4224 ]])

In [11]:
# Run it
result = greedy_map_dpp(L_quality, k=2)
print(f"Greedy MAP result: {[f'{genre[i]} ({relevance[i]})' for i in result]}")

Greedy MAP result: ['Action (0.9)', 'Comedy (0.7)']


In [33]:
def youtube_dpp_kernel(relevance, embeddings, alpha=0.5, sigma=1.0):
    N = len(relevance)
    L = np.zeros((N, N))
    
    # Compute pairwise distances
    # D_ij = ||embedding_i - embedding_j||²
    for i in range(N):
        for j in range(N):
            if i == j:
                # Diagonal: quality squared
                
                L[i, i] = relevance[i] ** 2
            else:
                # Off-diagonal: scaled similarity with RBF kernel
                D_ij = np.sum((embeddings[i] - embeddings[j]) ** 2)
                similarity = np.exp(-D_ij / (2 * sigma**2))
                L[i, j] = alpha * relevance[i] * relevance[j] * similarity
    
    return L

# Build kernel with YouTube's approach
L_youtube = youtube_dpp_kernel(relevance, embeddings, alpha=0.5, sigma=1.0)
print("YouTube DPP kernel (α=0.5, σ=1.0):")
print(np.round(L_youtube, 3))

YouTube DPP kernel (α=0.5, σ=1.0):
[[0.81  0.379 0.349 0.116 0.099 0.117]
 [0.379 0.722 0.337 0.132 0.103 0.126]
 [0.349 0.337 0.64  0.135 0.116 0.133]
 [0.116 0.132 0.135 0.49  0.077 0.136]
 [0.099 0.103 0.116 0.077 0.36  0.116]
 [0.117 0.126 0.133 0.136 0.116 0.25 ]]


In [40]:
alpha=0.5
sigma=1.0

D_ij = np.sum((embeddings - embeddings) ** 2, axis=-1)
similarity = np.exp(-D_ij / (2 * sigma**2))
alpha * relevance * relevance * similarity

array([0.405  , 0.36125, 0.32   , 0.245  , 0.18   , 0.125  ])

In [45]:
embeddings

array([[1. , 0. , 0. ],
       [0.9, 0.1, 0. ],
       [0.8, 0.1, 0.1],
       [0. , 1. , 0. ],
       [0. , 0. , 1. ],
       [0.1, 0.5, 0.5]])

In [46]:
np.expand_dims(embeddings, axis=0)

array([[[1. , 0. , 0. ],
        [0.9, 0.1, 0. ],
        [0.8, 0.1, 0.1],
        [0. , 1. , 0. ],
        [0. , 0. , 1. ],
        [0.1, 0.5, 0.5]]])

In [44]:
embeddings[:, None]

array([[[1. , 0. , 0. ]],

       [[0.9, 0.1, 0. ]],

       [[0.8, 0.1, 0.1]],

       [[0. , 1. , 0. ]],

       [[0. , 0. , 1. ]],

       [[0.1, 0.5, 0.5]]])

In [51]:
(np.expand_dims(embeddings, axis=0) - np.expand_dims(embeddings.T, axis=1)) ** 2

ValueError: operands could not be broadcast together with shapes (1,6,3) (3,1,6) 

In [41]:
np.sum((embeddings - embeddings) ** 2, axis=-1)

array([0., 0., 0., 0., 0., 0.])

In [38]:
D_ij = np.sum((embeddings[0] - embeddings[1]) ** 2)
similarity = np.exp(-D_ij / (2 * sigma**2))
alpha * relevance[0] * relevance[1] * similarity

np.float64(0.3786940614090568)

In [106]:
relevance[0] * relevance[1]

np.float64(0.765)

In [107]:
relevance * relevance

array([0.81  , 0.7225, 0.64  , 0.49  , 0.36  , 0.25  ])

In [124]:
relevance

array([0.9 , 0.85, 0.8 , 0.7 , 0.6 , 0.5 ])

In [123]:
np.expand_dims(relevance, -1) * np.tile(relevance, (relevance.shape[0], 1))

array([[0.81  , 0.765 , 0.72  , 0.63  , 0.54  , 0.45  ],
       [0.765 , 0.7225, 0.68  , 0.595 , 0.51  , 0.425 ],
       [0.72  , 0.68  , 0.64  , 0.56  , 0.48  , 0.4   ],
       [0.63  , 0.595 , 0.56  , 0.49  , 0.42  , 0.35  ],
       [0.54  , 0.51  , 0.48  , 0.42  , 0.36  , 0.3   ],
       [0.45  , 0.425 , 0.4   , 0.35  , 0.3   , 0.25  ]])

In [131]:
relevance ** 2

array([0.81  , 0.7225, 0.64  , 0.49  , 0.36  , 0.25  ])

In [115]:
relevance.repeat(relevance.shape[0], axis=0)

array([0.9 , 0.9 , 0.9 , 0.9 , 0.9 , 0.9 , 0.85, 0.85, 0.85, 0.85, 0.85,
       0.85, 0.8 , 0.8 , 0.8 , 0.8 , 0.8 , 0.8 , 0.7 , 0.7 , 0.7 , 0.7 ,
       0.7 , 0.7 , 0.6 , 0.6 , 0.6 , 0.6 , 0.6 , 0.6 , 0.5 , 0.5 , 0.5 ,
       0.5 , 0.5 , 0.5 ])

In [112]:
relevance.repeat(relevance.shape[0], axis=0).reshape((relevance.shape[0], relevance.shape[0]))

array([[0.9 , 0.9 , 0.9 , 0.9 , 0.9 , 0.9 ],
       [0.85, 0.85, 0.85, 0.85, 0.85, 0.85],
       [0.8 , 0.8 , 0.8 , 0.8 , 0.8 , 0.8 ],
       [0.7 , 0.7 , 0.7 , 0.7 , 0.7 , 0.7 ],
       [0.6 , 0.6 , 0.6 , 0.6 , 0.6 , 0.6 ],
       [0.5 , 0.5 , 0.5 , 0.5 , 0.5 , 0.5 ]])

In [57]:
len(embeddings)

6

In [91]:
np.sum((embeddings[0] - embeddings[1]) ** 2)

np.float64(0.019999999999999997)

In [101]:
np.exp(-np.sum((embeddings[0] - embeddings[1]) ** 2) / (2 * sigma**2))

np.float64(0.9900498337491681)

In [105]:
alpha * relevance[0] * relevance[1] * np.exp(-np.sum((embeddings[0] - embeddings[1]) ** 2) / (2 * sigma**2))

np.float64(0.3786940614090568)

In [56]:
(embeddings[0] - embeddings[1]) ** 2

array([0.01, 0.01, 0.  ])

In [324]:
embeddings.repeat(embeddings.shape[0], axis=0).reshape((embeddings.shape[0], embeddings.shape[0], embeddings.shape[1]))

array([[[1. , 0. , 0. ],
        [1. , 0. , 0. ],
        [1. , 0. , 0. ],
        [1. , 0. , 0. ],
        [1. , 0. , 0. ],
        [1. , 0. , 0. ]],

       [[0.9, 0.1, 0. ],
        [0.9, 0.1, 0. ],
        [0.9, 0.1, 0. ],
        [0.9, 0.1, 0. ],
        [0.9, 0.1, 0. ],
        [0.9, 0.1, 0. ]],

       [[0.8, 0.1, 0.1],
        [0.8, 0.1, 0.1],
        [0.8, 0.1, 0.1],
        [0.8, 0.1, 0.1],
        [0.8, 0.1, 0.1],
        [0.8, 0.1, 0.1]],

       [[0. , 1. , 0. ],
        [0. , 1. , 0. ],
        [0. , 1. , 0. ],
        [0. , 1. , 0. ],
        [0. , 1. , 0. ],
        [0. , 1. , 0. ]],

       [[0. , 0. , 1. ],
        [0. , 0. , 1. ],
        [0. , 0. , 1. ],
        [0. , 0. , 1. ],
        [0. , 0. , 1. ],
        [0. , 0. , 1. ]],

       [[0.1, 0.5, 0.5],
        [0.1, 0.5, 0.5],
        [0.1, 0.5, 0.5],
        [0.1, 0.5, 0.5],
        [0.1, 0.5, 0.5],
        [0.1, 0.5, 0.5]]])

In [336]:
embeddings

array([[1. , 0. , 0. ],
       [0.9, 0.1, 0. ],
       [0.8, 0.1, 0.1],
       [0. , 1. , 0. ],
       [0. , 0. , 1. ],
       [0.1, 0.5, 0.5]])

In [337]:
np.sum(embeddings ** 2, axis=-1)

array([1.  , 0.82, 0.66, 1.  , 1.  , 0.51])

In [340]:
np.matmul(embeddings, embeddings.T) * 2

array([[2.  , 1.8 , 1.6 , 0.  , 0.  , 0.2 ],
       [1.8 , 1.64, 1.46, 0.2 , 0.  , 0.28],
       [1.6 , 1.46, 1.32, 0.2 , 0.2 , 0.36],
       [0.  , 0.2 , 0.2 , 2.  , 0.  , 1.  ],
       [0.  , 0.  , 0.2 , 0.  , 2.  , 1.  ],
       [0.2 , 0.28, 0.36, 1.  , 1.  , 1.02]])

In [342]:
np.matmul(embeddings, embeddings.T) * 2

array([[2.  , 1.8 , 1.6 , 0.  , 0.  , 0.2 ],
       [1.8 , 1.64, 1.46, 0.2 , 0.  , 0.28],
       [1.6 , 1.46, 1.32, 0.2 , 0.2 , 0.36],
       [0.  , 0.2 , 0.2 , 2.  , 0.  , 1.  ],
       [0.  , 0.  , 0.2 , 0.  , 2.  , 1.  ],
       [0.2 , 0.28, 0.36, 1.  , 1.  , 1.02]])

In [339]:
np.expand_dims(np.sum(embeddings ** 2, axis=-1), 1) + np.expand_dims(np.sum(embeddings ** 2, axis=-1), axis=0)

array([[2.  , 1.82, 1.66, 2.  , 2.  , 1.51],
       [1.82, 1.64, 1.48, 1.82, 1.82, 1.33],
       [1.66, 1.48, 1.32, 1.66, 1.66, 1.17],
       [2.  , 1.82, 1.66, 2.  , 2.  , 1.51],
       [2.  , 1.82, 1.66, 2.  , 2.  , 1.51],
       [1.51, 1.33, 1.17, 1.51, 1.51, 1.02]])

In [345]:
np.expand_dims(np.sum(embeddings ** 2, axis=-1), -1)

array([[1.  ],
       [0.82],
       [0.66],
       [1.  ],
       [1.  ],
       [0.51]])

In [347]:
embeddings

array([[1. , 0. , 0. ],
       [0.9, 0.1, 0. ],
       [0.8, 0.1, 0.1],
       [0. , 1. , 0. ],
       [0. , 0. , 1. ],
       [0.1, 0.5, 0.5]])

In [346]:
np.expand_dims(np.sum(embeddings ** 2, axis=-1), axis=0)

array([[1.  , 0.82, 0.66, 1.  , 1.  , 0.51]])

In [349]:
np.sum(embeddings ** 2, axis=-1)

array([1.  , 0.82, 0.66, 1.  , 1.  , 0.51])

In [344]:
np.expand_dims(np.sum(embeddings ** 2, axis=-1), -1) + np.expand_dims(np.sum(embeddings ** 2, axis=-1), axis=0) - np.matmul(embeddings, embeddings.T) * 2

array([[0.  , 0.02, 0.06, 2.  , 2.  , 1.31],
       [0.02, 0.  , 0.02, 1.62, 1.82, 1.05],
       [0.06, 0.02, 0.  , 1.46, 1.46, 0.81],
       [2.  , 1.62, 1.46, 0.  , 2.  , 0.51],
       [2.  , 1.82, 1.46, 2.  , 0.  , 0.51],
       [1.31, 1.05, 0.81, 0.51, 0.51, 0.  ]])

In [323]:
D = np.sum((embeddings - embeddings.repeat(embeddings.shape[0], axis=0).reshape((embeddings.shape[0], embeddings.shape[0], embeddings.shape[1]))) ** 2, axis=-1)
D

array([[0.  , 0.02, 0.06, 2.  , 2.  , 1.31],
       [0.02, 0.  , 0.02, 1.62, 1.82, 1.05],
       [0.06, 0.02, 0.  , 1.46, 1.46, 0.81],
       [2.  , 1.62, 1.46, 0.  , 2.  , 0.51],
       [2.  , 1.82, 1.46, 2.  , 0.  , 0.51],
       [1.31, 1.05, 0.81, 0.51, 0.51, 0.  ]])

In [104]:
sim = np.exp(-D / (2 * sigma**2))
sim

array([[1.        , 0.99004983, 0.97044553, 0.36787944, 0.36787944,
        0.51944206],
       [0.99004983, 1.        , 0.99004983, 0.44485807, 0.40252422,
        0.59155536],
       [0.97044553, 0.99004983, 1.        , 0.48190899, 0.48190899,
        0.66697681],
       [0.36787944, 0.44485807, 0.48190899, 1.        , 0.36787944,
        0.7749165 ],
       [0.36787944, 0.40252422, 0.48190899, 0.36787944, 1.        ,
        0.7749165 ],
       [0.51944206, 0.59155536, 0.66697681, 0.7749165 , 0.7749165 ,
        1.        ]])

In [328]:
np.expand_dims(relevance, -1) * np.tile(relevance, (relevance.shape[0], 1))

array([[0.81  , 0.765 , 0.72  , 0.63  , 0.54  , 0.45  ],
       [0.765 , 0.7225, 0.68  , 0.595 , 0.51  , 0.425 ],
       [0.72  , 0.68  , 0.64  , 0.56  , 0.48  , 0.4   ],
       [0.63  , 0.595 , 0.56  , 0.49  , 0.42  , 0.35  ],
       [0.54  , 0.51  , 0.48  , 0.42  , 0.36  , 0.3   ],
       [0.45  , 0.425 , 0.4   , 0.35  , 0.3   , 0.25  ]])

In [329]:
np.expand_dims(relevance, -1) * np.expand_dims(relevance, 0)

array([[0.81  , 0.765 , 0.72  , 0.63  , 0.54  , 0.45  ],
       [0.765 , 0.7225, 0.68  , 0.595 , 0.51  , 0.425 ],
       [0.72  , 0.68  , 0.64  , 0.56  , 0.48  , 0.4   ],
       [0.63  , 0.595 , 0.56  , 0.49  , 0.42  , 0.35  ],
       [0.54  , 0.51  , 0.48  , 0.42  , 0.36  , 0.3   ],
       [0.45  , 0.425 , 0.4   , 0.35  , 0.3   , 0.25  ]])

In [128]:
big_L = alpha * np.expand_dims(relevance, -1) * np.tile(relevance, (relevance.shape[0], 1)) * sim
big_L

array([[0.405     , 0.37869406, 0.34936039, 0.11588202, 0.09932745,
        0.11687446],
       [0.37869406, 0.36125   , 0.33661694, 0.13234527, 0.10264368,
        0.12570551],
       [0.34936039, 0.33661694, 0.32      , 0.13493452, 0.11565816,
        0.13339536],
       [0.11588202, 0.13234527, 0.13493452, 0.245     , 0.07725468,
        0.13561039],
       [0.09932745, 0.10264368, 0.11565816, 0.07725468, 0.18      ,
        0.11623747],
       [0.11687446, 0.12570551, 0.13339536, 0.13561039, 0.11623747,
        0.125     ]])

In [137]:
diag_ind = np.diag_indices_from(big_L)

big_L[diag_ind] = (np.expand_dims(relevance, -1) * np.tile(relevance, (relevance.shape[0], 1)))[diag_ind]
big_L

array([[0.81      , 0.37869406, 0.34936039, 0.11588202, 0.09932745,
        0.11687446],
       [0.37869406, 0.7225    , 0.33661694, 0.13234527, 0.10264368,
        0.12570551],
       [0.34936039, 0.33661694, 0.64      , 0.13493452, 0.11565816,
        0.13339536],
       [0.11588202, 0.13234527, 0.13493452, 0.49      , 0.07725468,
        0.13561039],
       [0.09932745, 0.10264368, 0.11565816, 0.07725468, 0.36      ,
        0.11623747],
       [0.11687446, 0.12570551, 0.13339536, 0.13561039, 0.11623747,
        0.25      ]])

In [126]:
L_youtube

array([[0.81      , 0.37869406, 0.34936039, 0.11588202, 0.09932745,
        0.11687446],
       [0.37869406, 0.7225    , 0.33661694, 0.13234527, 0.10264368,
        0.12570551],
       [0.34936039, 0.33661694, 0.64      , 0.13493452, 0.11565816,
        0.13339536],
       [0.11588202, 0.13234527, 0.13493452, 0.49      , 0.07725468,
        0.13561039],
       [0.09932745, 0.10264368, 0.11565816, 0.07725468, 0.36      ,
        0.11623747],
       [0.11687446, 0.12570551, 0.13339536, 0.13561039, 0.11623747,
        0.25      ]])

In [138]:
big_L - L_youtube

array([[0., 0., 0., 0., 0., 0.],
       [0., 0., 0., 0., 0., 0.],
       [0., 0., 0., 0., 0., 0.],
       [0., 0., 0., 0., 0., 0.],
       [0., 0., 0., 0., 0., 0.],
       [0., 0., 0., 0., 0., 0.]])

In [92]:
embeddings.repeat(embeddings.shape[0], axis=0).reshape((embeddings.shape[0], embeddings.shape[0], embeddings.shape[1])).shape

(6, 6, 3)

In [94]:
embeddings.repeat(embeddings.shape[0], axis=0).reshape((embeddings.shape[0], embeddings.shape[0], embeddings.shape[1]))

array([[[1. , 0. , 0. ],
        [1. , 0. , 0. ],
        [1. , 0. , 0. ],
        [1. , 0. , 0. ],
        [1. , 0. , 0. ],
        [1. , 0. , 0. ]],

       [[0.9, 0.1, 0. ],
        [0.9, 0.1, 0. ],
        [0.9, 0.1, 0. ],
        [0.9, 0.1, 0. ],
        [0.9, 0.1, 0. ],
        [0.9, 0.1, 0. ]],

       [[0.8, 0.1, 0.1],
        [0.8, 0.1, 0.1],
        [0.8, 0.1, 0.1],
        [0.8, 0.1, 0.1],
        [0.8, 0.1, 0.1],
        [0.8, 0.1, 0.1]],

       [[0. , 1. , 0. ],
        [0. , 1. , 0. ],
        [0. , 1. , 0. ],
        [0. , 1. , 0. ],
        [0. , 1. , 0. ],
        [0. , 1. , 0. ]],

       [[0. , 0. , 1. ],
        [0. , 0. , 1. ],
        [0. , 0. , 1. ],
        [0. , 0. , 1. ],
        [0. , 0. , 1. ],
        [0. , 0. , 1. ]],

       [[0.1, 0.5, 0.5],
        [0.1, 0.5, 0.5],
        [0.1, 0.5, 0.5],
        [0.1, 0.5, 0.5],
        [0.1, 0.5, 0.5],
        [0.1, 0.5, 0.5]]])

In [95]:
(embeddings - embeddings.repeat(embeddings.shape[0], axis=0).reshape((embeddings.shape[0], embeddings.shape[0], embeddings.shape[1]))) ** 2

array([[[0.  , 0.  , 0.  ],
        [0.01, 0.01, 0.  ],
        [0.04, 0.01, 0.01],
        [1.  , 1.  , 0.  ],
        [1.  , 0.  , 1.  ],
        [0.81, 0.25, 0.25]],

       [[0.01, 0.01, 0.  ],
        [0.  , 0.  , 0.  ],
        [0.01, 0.  , 0.01],
        [0.81, 0.81, 0.  ],
        [0.81, 0.01, 1.  ],
        [0.64, 0.16, 0.25]],

       [[0.04, 0.01, 0.01],
        [0.01, 0.  , 0.01],
        [0.  , 0.  , 0.  ],
        [0.64, 0.81, 0.01],
        [0.64, 0.01, 0.81],
        [0.49, 0.16, 0.16]],

       [[1.  , 1.  , 0.  ],
        [0.81, 0.81, 0.  ],
        [0.64, 0.81, 0.01],
        [0.  , 0.  , 0.  ],
        [0.  , 1.  , 1.  ],
        [0.01, 0.25, 0.25]],

       [[1.  , 0.  , 1.  ],
        [0.81, 0.01, 1.  ],
        [0.64, 0.01, 0.81],
        [0.  , 1.  , 1.  ],
        [0.  , 0.  , 0.  ],
        [0.01, 0.25, 0.25]],

       [[0.81, 0.25, 0.25],
        [0.64, 0.16, 0.25],
        [0.49, 0.16, 0.16],
        [0.01, 0.25, 0.25],
        [0.01, 0.25, 0.25],
        [0

In [65]:
np.tile(embeddings, (len(embeddings), 1))

array([[1. , 0. , 0. ],
       [0.9, 0.1, 0. ],
       [0.8, 0.1, 0.1],
       [0. , 1. , 0. ],
       [0. , 0. , 1. ],
       [0.1, 0.5, 0.5],
       [1. , 0. , 0. ],
       [0.9, 0.1, 0. ],
       [0.8, 0.1, 0.1],
       [0. , 1. , 0. ],
       [0. , 0. , 1. ],
       [0.1, 0.5, 0.5],
       [1. , 0. , 0. ],
       [0.9, 0.1, 0. ],
       [0.8, 0.1, 0.1],
       [0. , 1. , 0. ],
       [0. , 0. , 1. ],
       [0.1, 0.5, 0.5],
       [1. , 0. , 0. ],
       [0.9, 0.1, 0. ],
       [0.8, 0.1, 0.1],
       [0. , 1. , 0. ],
       [0. , 0. , 1. ],
       [0.1, 0.5, 0.5],
       [1. , 0. , 0. ],
       [0.9, 0.1, 0. ],
       [0.8, 0.1, 0.1],
       [0. , 1. , 0. ],
       [0. , 0. , 1. ],
       [0.1, 0.5, 0.5],
       [1. , 0. , 0. ],
       [0.9, 0.1, 0. ],
       [0.8, 0.1, 0.1],
       [0. , 1. , 0. ],
       [0. , 0. , 1. ],
       [0.1, 0.5, 0.5]])

In [68]:
np.expand_dims(embeddings, axis=0)

array([[[1. , 0. , 0. ],
        [0.9, 0.1, 0. ],
        [0.8, 0.1, 0.1],
        [0. , 1. , 0. ],
        [0. , 0. , 1. ],
        [0.1, 0.5, 0.5]]])

In [80]:
embeddings.repeat(embeddings.shape[0], axis=0).reshape((embeddings.shape[0], embeddings.shape[0], embeddings.shape[1]))

array([[[1. , 0. , 0. ],
        [1. , 0. , 0. ],
        [1. , 0. , 0. ],
        [1. , 0. , 0. ],
        [1. , 0. , 0. ],
        [1. , 0. , 0. ]],

       [[0.9, 0.1, 0. ],
        [0.9, 0.1, 0. ],
        [0.9, 0.1, 0. ],
        [0.9, 0.1, 0. ],
        [0.9, 0.1, 0. ],
        [0.9, 0.1, 0. ]],

       [[0.8, 0.1, 0.1],
        [0.8, 0.1, 0.1],
        [0.8, 0.1, 0.1],
        [0.8, 0.1, 0.1],
        [0.8, 0.1, 0.1],
        [0.8, 0.1, 0.1]],

       [[0. , 1. , 0. ],
        [0. , 1. , 0. ],
        [0. , 1. , 0. ],
        [0. , 1. , 0. ],
        [0. , 1. , 0. ],
        [0. , 1. , 0. ]],

       [[0. , 0. , 1. ],
        [0. , 0. , 1. ],
        [0. , 0. , 1. ],
        [0. , 0. , 1. ],
        [0. , 0. , 1. ],
        [0. , 0. , 1. ]],

       [[0.1, 0.5, 0.5],
        [0.1, 0.5, 0.5],
        [0.1, 0.5, 0.5],
        [0.1, 0.5, 0.5],
        [0.1, 0.5, 0.5],
        [0.1, 0.5, 0.5]]])

In [53]:
embeddings

array([[1. , 0. , 0. ],
       [0.9, 0.1, 0. ],
       [0.8, 0.1, 0.1],
       [0. , 1. , 0. ],
       [0. , 0. , 1. ],
       [0.1, 0.5, 0.5]])

In [52]:
X_norm = np.sum(embeddings ** 2, axis = -1)
alpha * np.exp(-sigma * (X_norm[:,None] + X_norm[None,:] - 2 * np.dot(embeddings, embeddings.T)))

array([[0.5       , 0.49009934, 0.47088227, 0.06766764, 0.06766764,
        0.13491003],
       [0.49009934, 0.5       , 0.49009934, 0.09894935, 0.08101288,
        0.17496887],
       [0.47088227, 0.49009934, 0.5       , 0.11611814, 0.11611814,
        0.22242903],
       [0.06766764, 0.09894935, 0.11611814, 0.5       , 0.06766764,
        0.30024779],
       [0.06766764, 0.08101288, 0.11611814, 0.06766764, 0.5       ,
        0.30024779],
       [0.13491003, 0.17496887, 0.22242903, 0.30024779, 0.30024779,
        0.5       ]])

In [139]:
def youtube_dpp_kernel_fr(relevance, embeddings, alpha=0.5, sigma=1.0):
    D = np.sum(
        (embeddings - embeddings.repeat(embeddings.shape[0], axis=0)
                    .reshape((embeddings.shape[0], embeddings.shape[0], embeddings.shape[1]))) ** 2
        , axis=-1)
    similarity = np.exp(-D / (2 * sigma**2))
    relevant_arr = np.expand_dims(relevance, -1) * np.tile(relevance, (relevance.shape[0], 1))
    L = alpha * relevant_arr * similarity
    
    diag_ind = np.diag_indices_from(L)
    L[diag_ind] = relevant_arr[diag_ind]
    
    return L

# Build kernel with YouTube's approach
L_youtube = youtube_dpp_kernel_fr(relevance, embeddings, alpha=0.5, sigma=1.0)
print("YouTube DPP kernel (α=0.5, σ=1.0):")
print(np.round(L_youtube, 3))

YouTube DPP kernel (α=0.5, σ=1.0):
[[0.81  0.379 0.349 0.116 0.099 0.117]
 [0.379 0.722 0.337 0.132 0.103 0.126]
 [0.349 0.337 0.64  0.135 0.116 0.133]
 [0.116 0.132 0.135 0.49  0.077 0.136]
 [0.099 0.103 0.116 0.077 0.36  0.116]
 [0.117 0.126 0.133 0.136 0.116 0.25 ]]


In [141]:
print("YouTube Windowed DPP (k=2, α=0.75, σ=1.0)")
print("=" * 70)
# We'll manually trace the first few iterations
W = list(range(len(relevance)))
R = []
k_window = 2
alpha, sigma = 0.75, 1.0
iteration = 1
while len(W) > 0 and iteration <= 2:  # Show first 3 iterations
    print(f"\nIteration {iteration}:")
    print(f"  Remaining pool W: {W}")
    print(f"  Pool genres: {[genre[i] for i in W]}")
    
    # Build kernel for current pool
    rel_W = relevance[W]
    emb_W = embeddings[W]
    L = youtube_dpp_kernel_fr(rel_W, emb_W, alpha=alpha, sigma=sigma)
    
    # Select k items
    window_size = min(k_window, len(W))
    M = greedy_map_dpp(L, k=window_size)
    selected_items = [W[i] for i in M]
    
    print(f"  Selected from pool: {M} (pool indices)")
    print(f"  Maps to original: {selected_items}")
    print(f"  Genres: {[genre[i] for i in selected_items]}")
    print(f"  Relevance: {[relevance[i] for i in selected_items]}")
    
    R.extend(selected_items)
    W = [w for w in W if w not in selected_items]
    
    iteration += 1
print(f"\n{'─'*70}")
print(f"Final ranking R: {R}")
print(f"Genres: {[genre[i] for i in R]}")

YouTube Windowed DPP (k=2, α=0.75, σ=1.0)

Iteration 1:
  Remaining pool W: [0, 1, 2, 3, 4, 5]
  Pool genres: ['Action', 'Action', 'Action', 'Comedy', 'Documentary', 'Mixed comedy/documentary']
  Selected from pool: [0, 3] (pool indices)
  Maps to original: [0, 3]
  Genres: ['Action', 'Comedy']
  Relevance: [np.float64(0.9), np.float64(0.7)]

Iteration 2:
  Remaining pool W: [1, 2, 4, 5]
  Pool genres: ['Action', 'Action', 'Documentary', 'Mixed comedy/documentary']
  Selected from pool: [0, 2] (pool indices)
  Maps to original: [1, 4]
  Genres: ['Action', 'Documentary']
  Relevance: [np.float64(0.85), np.float64(0.6)]

──────────────────────────────────────────────────────────────────────
Final ranking R: [0, 3, 1, 4]
Genres: ['Action', 'Comedy', 'Action', 'Documentary']


In [14]:
3
2

1 - 2

-1

In [322]:
def youtube_dpp_ranking(relevance, embeddings, k, output_size, alpha=0.5, sigma=1.0):
    W = list(range(len(relevance)))  # Remaining candidate indices
    R = []  # Final ranked list
    
    remaining = output_size
    while len(W) > 0 and remaining > 0:        
        # Build kernel for current candidate pool
        rel_W = relevance[W]
        emb_W = embeddings[W]
        print("Emb W:", emb_W, emb_W.shape[1])
        L = youtube_dpp_kernel_fr(rel_W, emb_W, alpha=alpha, sigma=sigma)
        
        # Select up to k items from current pool
        window_size = min(k, remaining)
        M = greedy_map_dpp_fr(L, k=window_size)
        
        # Map back to original indices
        selected_items = [W[i] for i in M]
        R.extend(selected_items)
        
        # Remove selected items from candidate pool
        W = [w for w in W if w not in selected_items]
        remaining -= window_size
    
    return R

relevance = np.array([0.9, 0.85, 0.8, 0.7, 0.6, 0.5])
genre = ['Action', 'Action', 'Action', 'Comedy', 'Documentary', 'Mixed comedy/documentary']
embeddings = np.array([
    [1.0, 0.0, 0.0],   # video 0: pure action
    [0.9, 0.1, 0.0],   # video 1: action (very similar to 0)
    [0.8, 0.1, 0.1],   # video 2: action (also very similar to 0)
    [0.0, 1.0, 0.0],   # video 3: comedy
    [0.0, 0.0, 1.0],   # video 4: documentary
    [0.1, 0.5, 0.5],   # video 5: mixed
])

R = youtube_dpp_ranking(relevance, embeddings, k=3, output_size=3, alpha=0.75)
print(f"Final ranking R: {R}")
print(f"Genres: {[genre[i] for i in R]}")

Emb W: [[1.  0.  0. ]
 [0.9 0.1 0. ]
 [0.8 0.1 0.1]
 [0.  1.  0. ]
 [0.  0.  1. ]
 [0.1 0.5 0.5]] 3
0
0.81

3
0.36668555217190596

4
0.11625464440410205

Final ranking R: [0, 3, 4]
Genres: ['Action', 'Comedy', 'Documentary']
